In [0]:
import pandas as pd
from pyspark.sql import functions as F

In [0]:
schema = 'finance'
table_name = 'dim_taxonomy'

In [0]:
dbutils.widgets.text("year", "", "GAAP Version Year")
gaap_year_to_process = dbutils.widgets.get("year")

dbutils.widgets.text("target_catalog", "", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")

In [0]:
df = spark.table("operations.finance_staging.dim_taxonomy_staging").filter(F.col("linkrole")=="http://fasb.org/us-gaap/role/statement/StatementOfFinancialPositionClassified").toPandas()

fact = spark.table("operations.finance_staging.fact_staging_financial_statement").select("terse_label").distinct().toPandas()

leaf_nodes = set(fact['terse_label'])

# parent_map = dict(zip(df["child_label"],df['parent_label']))

parent_map = {
    (row["child_label"], row["gaap_version"], row["linkrole"]): row["parent_label"]
    for _, row in df.iterrows()
}


In [0]:
print(df.columns)

In [0]:
# def build_path(child, parent_map, max_depth = 50):
#     path = []
#     current = child
#     visited = set()

#     for _ in range(max_depth):
#         if current is None or current in visited:
#             break

#         path.append(current)
#         visited.add(current)
#         current = parent_map.get(current)
    
#     return path[::-1]

# paths = []

# for leaf in leaf_nodes:
#     path = build_path(leaf, parent_map)
#     paths.append({
#         "leaf_node": leaf,
#         "path": path
#     })

# paths_df = pd.DataFrame(paths)

# max_depth = paths_df["path"].apply(len).max()

# for i in range(max_depth):
#     paths_df[f"level_{i}"] = paths_df["path"].apply(
#         lambda x: x[i] if i < len(x) else None
#     )

# label_map = dict(zip(df["child_label"], df["child_label"]))  # or use another column

# for col in [c for c in paths_df.columns if c.startswith("level_")]:
#     paths_df[col] = paths_df[col].map(label_map)

# define linkrole once
linkrole_value = "http://fasb.org/us-gaap/role/statement/StatementOfFinancialPositionClassified"

# build parent map
parent_map = {
    (row["child_label"], row["gaap_version"], row["linkrole"]): row["parent_label"]
    for _, row in df.iterrows()
}

# optimize version lookup
child_version_map = (
    df.groupby("child_label")["gaap_version"]
    .unique()
    .to_dict()
)

def build_path(child, gaap_version, linkrole, parent_map, max_depth=50):
    path = []
    current = child
    visited = set()

    for _ in range(max_depth):
        key = (current, gaap_version, linkrole)

        if current is None or key in visited:
            break

        path.append(current)
        visited.add(key)

        current = parent_map.get(key)

    return path[::-1]

paths = []

for leaf in leaf_nodes:
    versions = child_version_map.get(leaf, [])

    for version in versions:
        path = build_path(leaf, version, linkrole_value, parent_map)

        paths.append({
            "leaf_node": leaf,
            "gaap_version": version,
            "path": path
        })

paths_df = pd.DataFrame(paths)

max_depth = paths_df["path"].apply(len).max()

for i in range(max_depth):
    paths_df[f"level_{i}"] = paths_df["path"].apply(
        lambda x: x[i] if i < len(x) else None
    )

label_map = dict(zip(df["child_label"], df["child_label"]))

for col in [c for c in paths_df.columns if c.startswith("level_")]:
    paths_df[col] = paths_df[col].map(label_map)

In [0]:
spark.createDataFrame(paths_df).createOrReplaceTempView('df')

final_df = spark.sql(f"""
select 
bigint(substr(xxhash64(concat_ws('|', leaf_node)), 1, 18)) AS terse_label_bigint_key
,sha2(concat_ws('|', leaf_node), 256) AS terse_label_key_hash
,bigint(substr(xxhash64(concat_ws('|', gaap_version)), 1, 18))  as gaap_version_bigint_key
,sha2(concat_ws('|', gaap_version), 256) as gaap_version_key_hash
,gaap_version as gaap_version
,leaf_node as terse_label
,level_1 as terse_label_level_1
,level_2 as terse_label_level_2
,level_3 as terse_label_level_3
,level_4 as terse_label_level_4
,level_5 as terse_label_level_5
,level_6 as terse_label_level_6
,level_7 as terse_label_level_7
,level_8 as terse_label_level_8
,level_9 as terse_label_level_9
,level_10 as terse_label_level_10
,level_11 as terse_label_level_11
from 
df
""")

In [0]:
final_df.write.mode("overwrite").saveAsTable(f"{target_catalog}.{schema}.{table_name}")